Bloque 1: Instalación de Dependencias

In [3]:
!pip install -q langchain langchain-groq langchain-community faiss-cpu python-dotenv langgraph

In [22]:
import os
from google.colab import userdata
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.embeddings import FakeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
import operator

# Carga segura de la API Key desde los secretos de Google Colab
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

Bloque 3: Base de Conocimiento RAG (Largo Plazo)

In [23]:
from langchain_core.embeddings import FakeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

documentos_institucionales = [
    "Póliza Vehicular (POL-V01): Cobertura total ante daños propios, responsabilidad civil de hasta 50000 USD y asistencia en ruta 24/7. Deducible estándar de 200 USD.",
    "Póliza de Hogar (POL-H02): Cobertura frente a incendios, sismos, robos y daños por rotura de tuberías. Incluye servicios de emergencia domiciliaria.",
    "Póliza de Vida (POL-V03): Cobertura por fallecimiento e invalidez total o parcial, con beneficios adicionales por hospitalización."
]

text_splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=0)
docs = text_splitter.create_documents(documentos_institucionales)

# Usamos FakeEmbeddings con una dimensión fija para evitar que busque modelos en OpenAI o Groq
embeddings = FakeEmbeddings(size=10)
vector_store = FAISS.from_documents(docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

Bloque 4: Definición de Herramientas (Consulta y Escritura)

In [24]:
@tool
def buscar_normativa_documental(consulta: str) -> str:
    """Recupera contexto semántico desde la base de conocimiento institucional de la aseguradora."""
    resultados = retriever.invoke(consulta)
    if resultados:
        return "\n".join([doc.page_content for doc in resultados])
    return "No se encontró información institucional relevante."

@tool
def consultar_estado_poliza(id_cliente: str) -> str:
    """Consulta los registros transaccionales de pólizas de los clientes."""
    base_datos = {
        "CLI-001": {"nombre": "Juan Pérez", "poliza": "POL-9981", "tipo": "Vehicular", "estado": "Vigente"},
        "CLI-002": {"nombre": "María Gómez", "poliza": "POL-4432", "tipo": "Hogar", "estado": "Pendiente de Pago"},
        "CLI-003": {"nombre": "Carlos Soto", "poliza": "POL-1120", "tipo": "Vida", "estado": "Vigente"}
    }
    cliente = base_datos.get(id_cliente)
    if cliente:
        return f"Cliente: {cliente['nombre']}, Póliza: {cliente['poliza']}, Tipo: {cliente['tipo']}, Estado: {cliente['estado']}."
    return f"No se encontró registro para el ID: {id_cliente}."

@tool
def registrar_modificacion_sistema(id_cliente: str, detalles_actualizacion: str) -> str:
    """Herramienta de escritura para registrar cambios o incidencias en el sistema de la organización."""
    return f"Éxito: Se ha registrado la actualización para el cliente {id_cliente} con detalle: '{detalles_actualizacion}'."

tools = [buscar_normativa_documental, consultar_estado_poliza, registrar_modificacion_sistema]
tool_node = ToolNode(tools)

Bloque 5: Modelo Groq y Orquestación con LangGraph y Memoria

In [25]:
from langchain_groq import ChatGroq

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# Cambiamos a un modelo estándar y ampliamente soportado en Groq
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0).bind_tools(tools)

def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def call_model(state: AgentState):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

# Compilación con memoria de corto plazo independiente (MemorySaver)
memory = MemorySaver()
app_agente = workflow.compile(checkpointer=memory)

Bloque 6: Pruebas y Validación con Sesión Persistente

In [26]:
config = {"configurable": {"thread_id": "sesion-evaluacion-ep2-groq-01"}}

# Turno 1: Búsqueda institucional (RAG)
print("--- TURNO 1: Consulta Institucional con Groq ---")
input_1 = {"messages": [HumanMessage(content="¿Qué cubren las pólizas vehiculares en Aseguradora del Pacífico?")]}
for event in app_agente.stream(input_1, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

# Turno 2: Consulta transaccional y validación de memoria conversacional
print("\n--- TURNO 2: Consulta Transaccional con Memoria (Groq) ---")
input_2 = {"messages": [HumanMessage(content="Excelente. Ahora revísame el estado del cliente con ID CLI-001.")]}
for event in app_agente.stream(input_2, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

--- TURNO 1: Consulta Institucional con Groq ---
================================ Human Message =================================

¿Qué cubren las pólizas vehiculares en Aseguradora del Pacífico?
================================== Ai Message ==================================
Tool Calls:
  buscar_normativa_documental (fc_cd898e2b-054c-4e7a-beeb-8a5836eec0ae)
 Call ID: fc_cd898e2b-054c-4e7a-beeb-8a5836eec0ae
  Args:
    consulta: Cobertura de pólizas vehiculares en Aseguradora del Pacífico
================================= Tool Message =================================
Name: buscar_normativa_documental

Póliza de Vida (POL-V03): Cobertura por fallecimiento e invalidez total o parcial, con beneficios adicionales por hospitalización.
================================== Ai Message ==================================

Para poder darte una respuesta precisa necesito saber a qué tipo de cobertura te refieres. En Aseguradora del Pacífico las pólizas vehiculares suelen incluir varias coberturas o